In [1]:
import torch, torchaudio, random, itertools
import torch.nn as nn
import torch.optim as optim
import numpy as np, pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE, HOP_SECONDS = 44100, 0.010
HOP, N_FFT, N_MELS = int(HOP_SECONDS * SAMPLE_RATE), 2048, 128

class OnsetCRNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.mel   = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        self.rnn = nn.GRU(128 * (N_MELS // 4), 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head, self.onset_head, self.offset_head = nn.Linear(256, 2), nn.Linear(256, 1), nn.Linear(256, 1)
    def forward(self, x):
        s = self.cnn(self.to_db(self.mel(x)))
        B, C, M, T = s.shape
        s, _ = self.rnn(s.reshape(B, C*M, T).transpose(1, 2))
        s = self.drop(s)
        return self.state_head(s), self.onset_head(s).squeeze(-1), self.offset_head(s).squeeze(-1)

ROOT = Path("/kaggle/input/datasets/alirahmatpour/vocalset-onset")

# ---- VocalSet (ext4 labels) ----
VS_ANNOT = ROOT / "annot_ext4_upload" / "annot_ext4_upload" / "annot"
VS_AUDIO = ROOT / "kaggle_upload" / "kaggle_upload" / "audio"
vs_csv   = {p.stem: p for p in VS_ANNOT.rglob("*.csv")}
vs_wavs  = [w for w in sorted(VS_AUDIO.rglob("*.wav")) if w.stem in vs_csv]

# ---- TONAS ----
TONAS = ROOT / "tonas_upload"
tonas_pairs = [(w, w.parent / (w.stem + ".notes.Corrected")) for w in sorted(TONAS.rglob("*.wav"))
               if not w.name.startswith("._") and (w.parent / (w.stem + ".notes.Corrected")).exists()]

# ---- SVNote1 (3 annotators) ----
SV = ROOT / "svnote_upload"
ANNOTATORS = ["dooyong", "gwangseok", "sanghee"]
def sv_key(name):
    stem = name.split(".")[0]
    return "_".join(p for p in stem.split("_") if not p.isdigit())
sv_wavs = {sv_key(w.name): w for w in sorted((SV / "audio").rglob("*.wav")) if not w.name.startswith("._")}
sv_ann  = {a: {sv_key(c.name): c for c in (SV / "annot" / a).rglob("*.csv") if not c.name.startswith("._")}
           for a in ANNOTATORS}
sv_keys = sorted(set(sv_wavs).intersection(*[set(sv_ann[a]) for a in ANNOTATORS]))

# ---- CSD (clean upload) ----
CSD = ROOT / "csd_upload"
csd_index = pd.read_csv(CSD / "index.csv")

# ---- checks ----
print("device  :", device)
print("VocalSet:", len(vs_wavs), "wav+csv pairs (expect 2684, clean filter comes in cell 2)")
print("TONAS   :", len(tonas_pairs), "clips (expect 72)")
print("SVNote1 :", len(sv_keys), "songs with all 3 annotators (expect 27)")
print("CSD     :", len(csd_index), "files (expect 109) | audio found:",
      all((CSD / "audio" / f"{f}.wav").exists() for f in csd_index.file))

device  : cuda
VocalSet: 2684 wav+csv pairs (expect 2684, clean filter comes in cell 2)
TONAS   : 72 clips (expect 72)
SVNote1 : 27 songs with all 3 annotators (expect 27)
CSD     : 109 files (expect 109) | audio found: True


In [2]:
# ============================== helpers ==============================
def load_wave(path):
    """audio -> mono, 44.1 kHz"""
    wf, sr = torchaudio.load(str(path))
    w = wf.mean(0)
    if sr != SAMPLE_RATE:
        w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
    return w

def make_item(wave, on, off, test_onsets=None, clip_to_audio=False):
    """
    one recording:
      on, off      -> training labels (seconds)
      test_onsets  -> list of onset lists used for scoring (one per annotator); default = [on]
    """
    on, off = np.asarray(on, float), np.asarray(off, float)
    tests = test_onsets if test_onsets is not None else [on]
    nfr = 1 + len(wave) // HOP
    frames = []
    for t in tests:
        f = (np.asarray(t, float) / HOP_SECONDS).astype(int)
        if clip_to_audio:
            f = f[f < nfr]
        frames.append(set(f))
    return {"wave": wave, "on": on, "off": off, "frames": frames}

DATA = {}

# ============================== VocalSet ==============================
# test  = singers female4 + male6
# train = 48 clean files from the other singers (same 48 as in the 4x4 notebook)
COLS = ["Sequence","Start","End","Duration","Type","AvgF0","MedF0","MinF0","MaxF0",
        "StdF0","AvgF0_inSTD","EstMIDI","GTNote","GTFreq","GTMIDI","Lyric",
        "GTDurName","GTDur","IntPrev","IntNext","Extra"]
VS_TEST_SINGERS = {"female4", "male6"}

def vs_singer(w):
    return w.parts[w.parts.index("audio") + 1]

def vs_load(w):
    wave = load_wave(w)
    ann = pd.read_csv(vs_csv[w.stem], skiprows=17, header=None, names=COLS)
    ann = ann[["Start", "End", "Type"]].dropna(subset=["Start", "End", "Type"])
    # clean filter: last label of ANY type (incl. final Rest) must match the audio length
    if not len(ann) or abs(ann["End"].max() / (len(wave) / SAMPLE_RATE) - 1.0) > 0.1:
        return None
    snd = ann[ann["Type"] == "Sound"]                                   # sung notes only
    x = make_item(wave, snd["Start"].values, snd["End"].values, clip_to_audio=True)
    x["singer"] = vs_singer(w)
    return x

vs_test = [x for x in (vs_load(w) for w in vs_wavs if vs_singer(w) in VS_TEST_SINGERS) if x is not None]

vs_cands = [w for w in vs_wavs if vs_singer(w) not in VS_TEST_SINGERS]
random.Random(0).shuffle(vs_cands)
vs_train = []
for w in vs_cands:
    if len(vs_train) >= 48:
        break
    x = vs_load(w)
    if x is not None:
        vs_train.append(x)
DATA["VocalSet"] = {"train": vs_train, "test": vs_test}

# ============================== TONAS ==============================
def tonas_singer(stem):
    return stem.split("_", 1)[1].split("-")[0] if "_" in stem else stem

t_items = []
for wav, notes in tonas_pairs:
    on, off = [], []
    for ln in notes.read_text(encoding="latin-1").strip().splitlines()[1:]:     # skip tuning line
        p = [x.strip() for x in ln.split(",")]
        if len(p) >= 2:
            on.append(float(p[0])); off.append(float(p[0]) + float(p[1]))
    x = make_item(load_wave(wav), on, off)
    x["singer"] = tonas_singer(wav.stem)
    t_items.append(x)

rng = random.Random(0)
t_singers = sorted({x["singer"] for x in t_items}); rng.shuffle(t_singers)
t_test_singers, n_test = set(), 0
for sg in t_singers:
    if n_test >= len(t_items) // 3:
        break
    t_test_singers.add(sg); n_test += sum(x["singer"] == sg for x in t_items)
DATA["TONAS"] = {"train": [x for x in t_items if x["singer"] not in t_test_singers],
                 "test":  [x for x in t_items if x["singer"] in t_test_singers]}

# ============================== SVNote1 ==============================
s_items = []
for k in sv_keys:
    tabs = [np.loadtxt(sv_ann[a][k], delimiter=",", ndmin=2) for a in ANNOTATORS]
    if len({len(t) for t in tabs}) == 1:
        cons = np.mean([t[:, :2] for t in tabs], axis=0)
    else:
        cons = tabs[0][:, :2]
        print("note count differs, using", ANNOTATORS[0], "for", k)
    x = make_item(load_wave(sv_wavs[k]), cons[:, 0], cons[:, 1], test_onsets=[t[:, 0] for t in tabs])
    x["singer"] = k.split("_")[0]
    s_items.append(x)

rng = random.Random(0)
men   = sorted({x["singer"] for x in s_items if x["singer"].startswith("man")})
women = sorted({x["singer"] for x in s_items if x["singer"].startswith("woman")})
rng.shuffle(men); rng.shuffle(women)
s_test_singers = set(men[:2] + women[:1])
DATA["SVNote1"] = {"train": [x for x in s_items if x["singer"] not in s_test_singers],
                   "test":  [x for x in s_items if x["singer"] in s_test_singers]}

# ============================== CSD ==============================
c_items = []
for r in csd_index.itertuples():
    lab = pd.read_csv(CSD / "annot" / f"{r.file}.csv")
    x = make_item(load_wave(CSD / "audio" / f"{r.file}.wav"), lab["start"].values, lab["end"].values)
    x["song"], x["lang"] = r.song, r.lang
    c_items.append(x)

rng = random.Random(0)
c_test_songs = set()
for lang in ["english", "korean"]:
    ss = sorted({x["song"] for x in c_items if x["lang"] == lang}); rng.shuffle(ss)
    c_test_songs |= set(ss[: len(ss) // 3])
DATA["CSD"] = {"train": [x for x in c_items if x["song"] not in c_test_songs],
               "test":  [x for x in c_items if x["song"] in c_test_songs]}

# ============================== checks ==============================
print("dataset  | train files | test files | test notes")
for name, d in DATA.items():
    print(f"{name:8s} | {len(d['train']):11d} | {len(d['test']):10d} | "
          f"{sum(len(x['frames'][0]) for x in d['test'])}")
print("expected : VocalSet 48 / 252 / 2834 | TONAS 48 / 24 / 1066 | SVNote1 18 / 9 / 432 | CSD 75 / 34 / 5922")

vs_sex = pd.Series(["female" if x["singer"].startswith("female") else "male" for x in vs_test]).value_counts()
print("\nVocalSet test by sex:", vs_sex.to_dict(), "(expect male 143 / female 109)")
print("SVNote1 test singers:", sorted(s_test_singers), "(expect man3, man5, woman1)")

key = {"VocalSet": "singer", "TONAS": "singer", "SVNote1": "singer", "CSD": "song"}
for name, d in DATA.items():
    overlap = {x[key[name]] for x in d["train"]} & {x[key[name]] for x in d["test"]}
    print(f"{name:8s} train/test overlap ({key[name]}): {len(overlap)} (must be 0)")

dataset  | train files | test files | test notes
VocalSet |          48 |        252 | 2834
TONAS    |          48 |         24 | 1066
SVNote1  |          18 |          9 | 432
CSD      |          75 |         34 | 5922
expected : VocalSet 48 / 252 / 2834 | TONAS 48 / 24 / 1066 | SVNote1 18 / 9 / 432 | CSD 75 / 34 / 5922

VocalSet test by sex: {'male': 143, 'female': 109} (expect male 143 / female 109)
SVNote1 test singers: ['man3', 'man5', 'woman1'] (expect man3, man5, woman1)
VocalSet train/test overlap (singer): 0 (must be 0)
TONAS    train/test overlap (singer): 0 (must be 0)
SVNote1  train/test overlap (singer): 0 (must be 0)
CSD      train/test overlap (song): 0 (must be 0)


In [3]:
# ============================== scoring (same as all earlier notebooks) ==============================
def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i - peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def f1_sets(pred_list, true_list, tol):
    tp = fp = fn = 0
    for pred, t_idx in zip(pred_list, true_list):
        matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi - ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(set(pred) - matched)
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

def evaluate_on(model, test_items, thr=0.8):
    """F1 at ±50 (main) and ±20 ms; if several annotators, average over them."""
    model.eval()
    with torch.no_grad():
        peaks = [pick_peaks(torch.sigmoid(model(x["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy(), thr)
                 for x in test_items]
    n_ann = len(test_items[0]["frames"])
    out = {}
    for tol_name, tol in [("50", 5), ("20", 2)]:
        out[f"F1_{tol_name}"] = np.mean([f1_sets(peaks, [x["frames"][j] for x in test_items], tol)
                                         for j in range(n_ann)])
    return out

# ============================== training (same as the 4x4 notebook) ==============================
SOFT_WIDTH = 2
def make_targets(x):
    nfr = 1 + x["wave"].shape[0] // HOP
    st = np.zeros(nfr, np.int64); on_t = np.zeros(nfr, np.float32); off_t = np.zeros(nfr, np.float32)
    def bump(arr, c):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH + 1):
            j = c + d
            if 0 <= j < nfr: arr[j] = max(arr[j], np.exp(-(d*d) / (2 * (SOFT_WIDTH/2)**2)))
    for a, b in zip(x["on"], x["off"]):
        fa, fb = int(a / HOP_SECONDS), int(b / HOP_SECONDS)
        st[fa:fb] = 1; bump(on_t, fa); bump(off_t, fb)
    return torch.tensor(st), torch.tensor(on_t), torch.tensor(off_t)

def train_scratch(train_items, seed, epochs=40, lr=1e-3, crop=1500):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tg = [make_targets(x) for x in train_items]
    model = OnsetCRNN().to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    ce, bce = nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0, device=device))
    idx = list(range(len(train_items)))
    for ep in range(epochs):
        model.train()
        for i in random.sample(idx, len(idx)):
            st, on, off = tg[i]
            nfr = len(st); a = np.random.randint(0, max(1, nfr - crop)); b = min(nfr, a + crop)
            ps, po, pf = model(train_items[i]["wave"][a*HOP : b*HOP].view(1, 1, -1).to(device))
            F = min(ps.shape[1], b - a)
            loss = (ce(ps[0, :F], st[a:a+F].to(device)) + bce(po[0, :F], on[a:a+F].to(device))
                    + bce(pf[0, :F], off[a:a+F].to(device)))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

# ============================== ALL TOGETHER ==============================
TEST_SETS = ["VocalSet", "TONAS", "SVNote1", "CSD"]
all_train = [x for name in TEST_SETS for x in DATA[name]["train"]]
print("training files:", {n: len(DATA[n]["train"]) for n in TEST_SETS}, "| total:", len(all_train), "(expect 189)")

rows_all = []
for k in range(3):
    print(f"\n[ALL TOGETHER | seed {k}] training on {len(all_train)} files ...")
    model = train_scratch(all_train, seed=k)
    torch.save(model.state_dict(), f"/kaggle/working/all_together_seed{k}.pt")
    for ts in TEST_SETS:
        r = evaluate_on(model, DATA[ts]["test"])
        rows_all.append({"setup": "all_together", "tested_on": ts, "seed": k, **r})
        print(f"  -> {ts:8s} | ±50 {r['F1_50']:.3f} | ±20 {r['F1_20']:.3f}")
    del model; torch.cuda.empty_cache()

all_res = pd.DataFrame(rows_all)
all_res.to_csv("/kaggle/working/onset_V3_all_together.csv", index=False)

print("\n=== ALL TOGETHER — mean ± std over 3 seeds ===")
for ts in TEST_SETS:
    g = all_res[all_res.tested_on == ts]
    print(f"{ts:8s} | ±50 {g.F1_50.mean():.3f} ± {g.F1_50.std(ddof=1):.3f}"
          f" | ±20 {g.F1_20.mean():.3f} ± {g.F1_20.std(ddof=1):.3f}")

training files: {'VocalSet': 48, 'TONAS': 48, 'SVNote1': 18, 'CSD': 75} | total: 189 (expect 189)

[ALL TOGETHER | seed 0] training on 189 files ...
  -> VocalSet | ±50 0.675 | ±20 0.484
  -> TONAS    | ±50 0.769 | ±20 0.536
  -> SVNote1  | ±50 0.962 | ±20 0.777
  -> CSD      | ±50 0.880 | ±20 0.768

[ALL TOGETHER | seed 1] training on 189 files ...
  -> VocalSet | ±50 0.671 | ±20 0.507
  -> TONAS    | ±50 0.753 | ±20 0.498
  -> SVNote1  | ±50 0.940 | ±20 0.719
  -> CSD      | ±50 0.891 | ±20 0.775

[ALL TOGETHER | seed 2] training on 189 files ...
  -> VocalSet | ±50 0.628 | ±20 0.479
  -> TONAS    | ±50 0.751 | ±20 0.512
  -> SVNote1  | ±50 0.957 | ±20 0.739
  -> CSD      | ±50 0.897 | ±20 0.784

=== ALL TOGETHER — mean ± std over 3 seeds ===
VocalSet | ±50 0.658 ± 0.026 | ±20 0.490 ± 0.015
TONAS    | ±50 0.758 ± 0.010 | ±20 0.515 ± 0.019
SVNote1  | ±50 0.953 ± 0.012 | ±20 0.745 ± 0.029
CSD      | ±50 0.890 ± 0.009 | ±20 0.776 ± 0.008


In [4]:
# ============================== LEAVE ONE OUT ==============================
rows_loo = []
for held_out in TEST_SETS:
    train_items = [x for name in TEST_SETS if name != held_out for x in DATA[name]["train"]]
    for k in range(3):
        print(f"\n[LEAVE OUT {held_out} | seed {k}] training on {len(train_items)} files "
              f"({', '.join(n for n in TEST_SETS if n != held_out)}) ...")
        model = train_scratch(train_items, seed=k)
        torch.save(model.state_dict(), f"/kaggle/working/loo_without_{held_out}_seed{k}.pt")
        r = evaluate_on(model, DATA[held_out]["test"])            # the UNSEEN dataset
        rows_loo.append({"setup": "leave_one_out", "tested_on": held_out, "seed": k, **r})
        print(f"  -> unseen {held_out:8s} | ±50 {r['F1_50']:.3f} | ±20 {r['F1_20']:.3f}")
        del model; torch.cuda.empty_cache()

loo_res = pd.DataFrame(rows_loo)
loo_res.to_csv("/kaggle/working/onset_V3_leave_one_out.csv", index=False)

print("\n=== LEAVE ONE OUT — score on the UNSEEN dataset, mean ± std over 3 seeds ===")
for ts in TEST_SETS:
    g = loo_res[loo_res.tested_on == ts]
    print(f"{ts:8s} | ±50 {g.F1_50.mean():.3f} ± {g.F1_50.std(ddof=1):.3f}"
          f" | ±20 {g.F1_20.mean():.3f} ± {g.F1_20.std(ddof=1):.3f}")


[LEAVE OUT VocalSet | seed 0] training on 141 files (TONAS, SVNote1, CSD) ...
  -> unseen VocalSet | ±50 0.480 | ±20 0.242

[LEAVE OUT VocalSet | seed 1] training on 141 files (TONAS, SVNote1, CSD) ...
  -> unseen VocalSet | ±50 0.456 | ±20 0.242

[LEAVE OUT VocalSet | seed 2] training on 141 files (TONAS, SVNote1, CSD) ...
  -> unseen VocalSet | ±50 0.456 | ±20 0.256

[LEAVE OUT TONAS | seed 0] training on 141 files (VocalSet, SVNote1, CSD) ...
  -> unseen TONAS    | ±50 0.616 | ±20 0.327

[LEAVE OUT TONAS | seed 1] training on 141 files (VocalSet, SVNote1, CSD) ...
  -> unseen TONAS    | ±50 0.389 | ±20 0.174

[LEAVE OUT TONAS | seed 2] training on 141 files (VocalSet, SVNote1, CSD) ...
  -> unseen TONAS    | ±50 0.544 | ±20 0.287

[LEAVE OUT SVNote1 | seed 0] training on 171 files (VocalSet, TONAS, CSD) ...
  -> unseen SVNote1  | ±50 0.823 | ±20 0.387

[LEAVE OUT SVNote1 | seed 1] training on 171 files (VocalSet, TONAS, CSD) ...
  -> unseen SVNote1  | ±50 0.761 | ±20 0.270

[LEAVE 